## Workshop - Deconstructing Language for AI Agents

Objective: By the end of this 90-minute workshop, you will understand and have implemented the core components of a Natural Language Processing (NLP) pipeline. You will learn how raw text is systematically processed to become useful, structured data for training an AI agent like Google's BARD.


## Team members



| Member | Student ID's |
|---|---|
| Ubaid Ullah |9110715 |
| Ved Vishal Patel | 9111628|
| Koushik Balne |  9087547 |
| Sukhchain Singh |  9111541|



### 1. Introduction: How Does an AI Learn Language?

When you ask an LLM, such as BARD, a question, it seems to just understand you. But that "understanding" is the result of a massive amount of pre-processing.

An AI doesn't read text like a human. It sees text as a sequence of numbers. To an AI, the words "run," "running," and "ran" are initially just different, unrelated data points. The goal of an NLP pipeline is to standardize and simplify text, transforming it into a structured format that a machine can learn patterns from.

Think of it as preparing ingredients for a recipe. You don't just throw a whole carrot into the pot. You wash it, peel it, and chop it. Our "ingredients" are words, and the "recipe" is the AI model we want to train. 

During this workshop, we'll learn the essential pre-processing steps.

### 2. The Raw Material: A Look at Our Text Data

For any NLP task, we need data. We'll be using the "20 Newsgroups" dataset, a classic collection of about 18,000 documents posted to 20 different Usenet newsgroups. It's perfect for us because it's real, messy text with a large vocabulary.

Let's start by loading the data and performing a quick Exploratory Data Analysis (EDA) to understand what we're working with.

Sample EDA: A First Glance at the Data
First, we need to install the necessary libraries. scikit-learn provides the dataset, nltk is our main NLP toolkit, and matplotlib helps us visualize our findings.

In [1]:
#
# Prerequisite installations for the workshop
# To set up the environment, run this script:
# !pip install scikit-learn nltk matplotlib
#
import nltk
import re
import os

# Warning: This download will copy files to your home directory.
# For example, on Linux, it will copy files to ~/.nltk_data.
# In Windows, it will copy files to C:\Users\YourAccount\AppData\Roaming
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')

# A better way to handle the download is to:
# Ensure 'punkt' is available and nltk_data path is set
nltk_data_path = os.path.join(os.getcwd(), 'nltk_data')
print("Downloading 'punkt' tokenizer...")
nltk.download('punkt', download_dir=nltk_data_path, force=True)
print("Downloading 'punkt_tab' tokenizer...")
nltk.download('punkt_tab', download_dir=nltk_data_path, force=True)
print("Downloading 'stopwords' tokenizer...")
nltk.download('stopwords', download_dir=nltk_data_path, force=True)
print("Downloading 'wordnet' tokenizer...")
nltk.download('wordnet', download_dir=nltk_data_path, force=True)

# Always append the custom nltk_data path (if not already present)
if nltk_data_path not in nltk.data.path:
    nltk.data.path.append(nltk_data_path)

# Debugging paths and contents
print("NLTK Data Paths:", nltk.data.path)
print("Contents of nltk_data:", os.listdir(nltk_data_path))


[nltk_data] Downloading package punkt to d:\Career\Conestoga\MLP\week
[nltk_data]     9\NLP_PipelineIntroductionWorkshop\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


[nltk_data] Downloading package punkt_tab to
[nltk_data]     d:\Career\Conestoga\MLP\week
[nltk_data]     9\NLP_PipelineIntroductionWorkshop\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     d:\Career\Conestoga\MLP\week
[nltk_data]     9\NLP_PipelineIntroductionWorkshop\nltk_data...


[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     d:\Career\Conestoga\MLP\week
[nltk_data]     9\NLP_PipelineIntroductionWorkshop\nltk_data...


NLTK Data Paths: ['C:\\Users\\Ubaid/nltk_data', 'd:\\Career\\Conestoga\\MLP\\week 9\\NLP_PipelineIntroductionWorkshop\\.venv\\nltk_data', 'd:\\Career\\Conestoga\\MLP\\week 9\\NLP_PipelineIntroductionWorkshop\\.venv\\share\\nltk_data', 'd:\\Career\\Conestoga\\MLP\\week 9\\NLP_PipelineIntroductionWorkshop\\.venv\\lib\\nltk_data', 'C:\\Users\\Ubaid\\AppData\\Roaming\\nltk_data', 'C:\\nltk_data', 'D:\\nltk_data', 'E:\\nltk_data', 'd:\\Career\\Conestoga\\MLP\\week 9\\NLP_PipelineIntroductionWorkshop\\nltk_data']
Contents of nltk_data: ['corpora', 'tokenizers']


Now, let's load the data and see what a raw document looks like.

In [2]:
# Load the dataset
from sklearn.datasets import fetch_20newsgroups
newsgroups_data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))

# Let's inspect a single document
print("--- RAW DOCUMENT ---")
sample_doc = newsgroups_data.data[12]
print(sample_doc)

# Let's get some basic stats
all_text = ' '.join(newsgroups_data.data)
words = all_text.split()
unique_words = set(words)

print(f"\n--- INITIAL STATS ---")
print(f"Number of documents: {len(newsgroups_data.data)}")
print(f"Total words (approx.): {len(words)}")
print(f"Total unique words (vocab size): {len(unique_words)}")

--- RAW DOCUMENT ---
930418

Do what thou wilt shall be the whole of the Law. [Honestly.]
The word of Sin is Restriction. [Would I kid you?]


Does one man's words encompass the majestic vision of thousands
of individuals?  Quoting a man is not the same as quoting the
Order.  Taken out of context, words can be interpreted much
differently than had one applied them within the confines of
their original expression.

I think this is the case regarding Hymenaeus Beta, Frater Superior 
of the Order to which I belong.  When he included that bit
from Merlinus X' he did us all a service.  He showed us the extremes
to which Order members have been known to go in their fervor.
I have little knowledge regarding Reuss' background, but surely
he was an unusual man, and he was an important force in the Order 
for many years.

Yet as people change so do Orders change, and while we look back
so carefully at the dirty laundry of O.T.O. remember that this is
only the surface skim and that many perspecti

You'll notice our vocabulary is massive! This is the complexity we need to reduce.

### 3. The NLP Pipeline: From Chaos to Structure

We will now build the core components of our pipeline. Each step feeds its output to the next, progressively refining the text.

#### Part 1: Tokenization - Creating the Building Blocks

Concept: Tokenization is the process of breaking down a stream of text into individual units, or tokens. These tokens are the fundamental building blocks for any further analysis. Usually, tokens are words, but they can also be sentences or characters.

Implementation: A Simple Tokenizer
We can start with a basic algorithm that splits text by spaces and removes punctuation.

In [3]:
import re

def simple_tokenizer(text):
  """
  A simple implementation of a word tokenizer.
  1. Splits text by non-word characters.
  2. Returns a list of word tokens.
  """
  # Use regular expression to find all sequences of word characters
  tokens = re.findall(r'\b\w+\b', text)
  return tokens

# Let's tokenize our sample document
raw_tokens = simple_tokenizer(sample_doc)
print("--- TOKENIZATION ---")
print(raw_tokens)

--- TOKENIZATION ---
['930418', 'Do', 'what', 'thou', 'wilt', 'shall', 'be', 'the', 'whole', 'of', 'the', 'Law', 'Honestly', 'The', 'word', 'of', 'Sin', 'is', 'Restriction', 'Would', 'I', 'kid', 'you', 'Does', 'one', 'man', 's', 'words', 'encompass', 'the', 'majestic', 'vision', 'of', 'thousands', 'of', 'individuals', 'Quoting', 'a', 'man', 'is', 'not', 'the', 'same', 'as', 'quoting', 'the', 'Order', 'Taken', 'out', 'of', 'context', 'words', 'can', 'be', 'interpreted', 'much', 'differently', 'than', 'had', 'one', 'applied', 'them', 'within', 'the', 'confines', 'of', 'their', 'original', 'expression', 'I', 'think', 'this', 'is', 'the', 'case', 'regarding', 'Hymenaeus', 'Beta', 'Frater', 'Superior', 'of', 'the', 'Order', 'to', 'which', 'I', 'belong', 'When', 'he', 'included', 'that', 'bit', 'from', 'Merlinus', 'X', 'he', 'did', 'us', 'all', 'a', 'service', 'He', 'showed', 'us', 'the', 'extremes', 'to', 'which', 'Order', 'members', 'have', 'been', 'known', 'to', 'go', 'in', 'their', 'ferv

> Talking Point: "Tokenization is the first and most critical step; it defines the discrete units of meaning—our 'words'—that the machine will learn from."

#### Part 2: Normalization & Stop Words Removal - Cleaning the Noise

Concept: Normalization is about standardizing tokens to treat different forms of a word as the same. The most common step is converting all text to lowercase. Why? Because we don't want the model to think "Apple" (the company) and "apple" (the fruit) are different purely based on capitalization if the context doesn't require it.

Concept: Stop words are extremely common words like "the," "a," "is," "in." They add grammatical structure for humans but often add little semantic value for a machine. Removing them helps the model focus on the words that carry the most meaning.

Implementation: Lowercasing and Filtering

In [4]:
from nltk.corpus import stopwords

def normalize_and_remove_stops(tokens):
    """
    1. Converts all tokens to lowercase (Normalization).
    2. Removes common English stop words.
    """
    # 1. Normalization: Convert to lowercase
    normalized_tokens = [token.lower() for token in tokens]

    # 2. Stop Words Removal
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [token for token in normalized_tokens if token not in stop_words]
    return filtered_tokens

# Process our tokens
cleaned_tokens = normalize_and_remove_stops(raw_tokens)
print("--- NORMALIZATION & STOP WORDS REMOVAL ---")
print(cleaned_tokens)

--- NORMALIZATION & STOP WORDS REMOVAL ---
['930418', 'thou', 'wilt', 'shall', 'whole', 'law', 'honestly', 'word', 'sin', 'restriction', 'would', 'kid', 'one', 'man', 'words', 'encompass', 'majestic', 'vision', 'thousands', 'individuals', 'quoting', 'man', 'quoting', 'order', 'taken', 'context', 'words', 'interpreted', 'much', 'differently', 'one', 'applied', 'within', 'confines', 'original', 'expression', 'think', 'case', 'regarding', 'hymenaeus', 'beta', 'frater', 'superior', 'order', 'belong', 'included', 'bit', 'merlinus', 'x', 'us', 'service', 'showed', 'us', 'extremes', 'order', 'members', 'known', 'go', 'fervor', 'little', 'knowledge', 'regarding', 'reuss', 'background', 'surely', 'unusual', 'man', 'important', 'force', 'order', 'many', 'years', 'yet', 'people', 'change', 'orders', 'change', 'look', 'back', 'carefully', 'dirty', 'laundry', 'remember', 'surface', 'skim', 'many', 'perspectives', 'encompassed', 'extend', 'beyond', 'one', 'individual', 'hope', 'show', 'much', 'room'

> Talking Point (Normalization): "By converting every token to lowercase, we prevent the model from treating the same word differently, drastically reducing the vocabulary size it needs to handle."

> Talking Point (Stop Words): "Removing stop words is like filtering out the static; it lets the model focus on the high-signal words that define the document's topic."

#### Part 3: Stemming - Finding the Root

Concept: Stemming is a process of reducing a word to its root or stem. It does this by applying a crude set of rules to chop off common prefixes or suffixes (affixes). For example, "running," "runs," and "ran" might all be stemmed to the base form "run." This process, known as affix stripping, is heuristic and can sometimes result in non-dictionary words (e.g., "studies" becomes "studi"). However, it's fast and effective for collapsing vocabulary.

Implementation: The Porter Stemmer
We will use the famous Porter Stemming algorithm, available in NLTK.

In [5]:
from nltk.stem import PorterStemmer

def stem_tokens(tokens):
    """
    Applies the Porter Stemming algorithm to a list of tokens.
    This is a form of affix stripping.
    """
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    return stemmed_tokens

# Apply stemming to our cleaned tokens
final_tokens = stem_tokens(cleaned_tokens)
print("--- STEMMING (AFFIX STRIPPING) ---")
print(final_tokens)

--- STEMMING (AFFIX STRIPPING) ---
['930418', 'thou', 'wilt', 'shall', 'whole', 'law', 'honestli', 'word', 'sin', 'restrict', 'would', 'kid', 'one', 'man', 'word', 'encompass', 'majest', 'vision', 'thousand', 'individu', 'quot', 'man', 'quot', 'order', 'taken', 'context', 'word', 'interpret', 'much', 'differ', 'one', 'appli', 'within', 'confin', 'origin', 'express', 'think', 'case', 'regard', 'hymenaeu', 'beta', 'frater', 'superior', 'order', 'belong', 'includ', 'bit', 'merlinu', 'x', 'us', 'servic', 'show', 'us', 'extrem', 'order', 'member', 'known', 'go', 'fervor', 'littl', 'knowledg', 'regard', 'reuss', 'background', 'sure', 'unusu', 'man', 'import', 'forc', 'order', 'mani', 'year', 'yet', 'peopl', 'chang', 'order', 'chang', 'look', 'back', 'care', 'dirti', 'laundri', 'rememb', 'surfac', 'skim', 'mani', 'perspect', 'encompass', 'extend', 'beyond', 'one', 'individu', 'hope', 'show', 'much', 'room', 'differ', 'opinion', 'within', 'order', 'perhap', 'test', 'limit', 'let', 'us', 'exami

> Talking Point (Stemming/Affix Stripping): "Stemming aggressively strips word endings to group related terms; it’s a powerful but blunt tool for consolidating our vocabulary down to its core concepts."

#### Part 4: The Final Result & The Big Picture

Let's put it all together and see the transformation.

In [6]:
# The full pipeline
def nlp_pipeline(text):
    tokens = simple_tokenizer(text)
    cleaned_tokens = normalize_and_remove_stops(tokens)
    final_tokens = stem_tokens(cleaned_tokens)
    return final_tokens

# Before
print("--- BEFORE PIPELINE ---")
print(sample_doc)

print("\n" + "="*50 + "\n")

# After
print("--- AFTER PIPELINE ---")
processed_doc = nlp_pipeline(sample_doc)
print(processed_doc)

# Let's re-evaluate our vocabulary size on the entire dataset
all_processed_tokens = nlp_pipeline(' '.join(newsgroups_data.data))
final_vocab_size = len(set(all_processed_tokens))

print(f"\n--- FINAL STATS ---")
print(f"Initial unique words: {len(unique_words)}")
print(f"Final unique words (vocab size): {final_vocab_size}")
print(f"Vocabulary Reduction: {((len(unique_words) - final_vocab_size) / len(unique_words)) * 100:.2f}%")

--- BEFORE PIPELINE ---
930418

Do what thou wilt shall be the whole of the Law. [Honestly.]
The word of Sin is Restriction. [Would I kid you?]


Does one man's words encompass the majestic vision of thousands
of individuals?  Quoting a man is not the same as quoting the
Order.  Taken out of context, words can be interpreted much
differently than had one applied them within the confines of
their original expression.

I think this is the case regarding Hymenaeus Beta, Frater Superior 
of the Order to which I belong.  When he included that bit
from Merlinus X' he did us all a service.  He showed us the extremes
to which Order members have been known to go in their fervor.
I have little knowledge regarding Reuss' background, but surely
he was an unusual man, and he was an important force in the Order 
for many years.

Yet as people change so do Orders change, and while we look back
so carefully at the dirty laundry of O.T.O. remember that this is
only the surface skim and that many perspe

### Part 5: Conclusion 

As you can see, we've taken unstructured, messy human language and converted it into a clean, standardized list of meaningful tokens. We've dramatically reduced the vocabulary size, making it much more efficient for an AI model to process.

This processed data is now ready to be converted into numerical vectors—a process called vectorization (e.g., TF-IDF or Word2Vec)—which is the final step before feeding it into a machine learning model to perform tasks like text classification, sentiment analysis, or even to train a foundational model like BARD.

### Part 6: Fun Activity: Ask BERT a Question!

So far, we've built an NLP pipeline from scratch. This is a fundamental skill, and it shows you the deliberate, step-by-step process of making text useful for machines.

Now, let's jump to the forefront of NLP and use a massive, pre-trained model: BERT (Bidirectional Encoder Representations from Transformers).

What is BERT?
Think of the pipeline we just built. Its goal was to simplify and standardize words. A model like BERT does something far more complex. During its "pre-training" on nearly the entire internet, it learned not just words, but the relationships between words.

Watch this video as an introduction to Transfoemrs to help you understand what BERT is, and where it came from:

[Transformer models and BERT model: Overview](http://www.youtube.com/watch?v=t45S_MwAcOw)


The key is its bidirectional nature. Unlike older models that read text left-to-right, BERT reads the entire sentence at once. This allows it to understand context. For example, it knows that "bank" in "river bank" is different from "bank" in "bank account." The pipeline we built would stem both to "bank" and lose that contextual meaning.

We can use the power of these pre-trained models for specific tasks without needing to train them ourselves. Let's try one of the most popular tasks: Extractive Question Answering.

The Goal: We will give the model a paragraph of text (the context) and a question. The model's job is to find and extract the exact span of text from the context that answers our question.

#### Implementation: The Hugging Face Pipeline 🚀

The transformers library from Hugging Face makes using models like BERT incredibly simple with a tool called pipeline. It handles all the complex tokenization and processing behind the scenes, so we can focus on the results.

In [7]:
#
# Prerequisite: Make sure you have transformers installed!
# Prerequisite: Install the necessary libraries
# The 'transformers' library requires either PyTorch or TensorFlow.
# We'll install torch here.
#
# !pip install transformers torch
#
from transformers import pipeline
import textwrap # Used for nice printing of the context

# 1. Create a Question-Answering pipeline
# This will download a default BERT-based model fine-tuned for this task.
# No authentication is needed!
qa_pipeline = pipeline("question-answering")

# 2. Define the context
# This is the body of text the model will read to find the answer.
context = """
BERT, which stands for Bidirectional Encoder Representations from Transformers,
is a large language model developed by Google. It was a major breakthrough in
the field of Natural Language Processing because it was the first model to look
at the full context of a word by examining the words that come before and after it.
This bidirectional capability allows it to grasp complex nuances in language.
The model was pre-trained on a massive amount of text data from sources like
Wikipedia and the BooksCorpus.
"""

# 3. Ask a question!
# The answer to this question must be present in the context above.
question = "What does BERT stand for?"

# 4. Get the answer
result = qa_pipeline(question=question, context=context)

# Let's print the results nicely
print("--- BERT Question-Answering ---")
print(f"❓ Question: {question}")
print(f"✅ Answer: '{result['answer']}'")
print(f"Confidence Score: {result['score']:.4f}")

# --- Now, it's your turn! ---
#
# Try asking different questions. For example:
# - "Who developed BERT?"
# - "What capability allows BERT to grasp nuances?"
# - "What was the model pre-trained on?"
#
# You can even change the context entirely to a paragraph about a
# topic you find interesting and ask questions about it!

print("\n--- Try it yourself! ---")
# Example of another question
my_question = "Who developed BERT?"
my_result = qa_pipeline(question=my_question, context=context)

print(f"❓ My Question: {my_question}")
print(f"✅ Answer: '{my_result['answer']}' with score {my_result['score']:.4f}")

d:\Career\Conestoga\MLP\week 9\NLP_PipelineIntroductionWorkshop\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


--- BERT Question-Answering ---
❓ Question: What does BERT stand for?
✅ Answer: 'Bidirectional Encoder Representations from Transformers'
Confidence Score: 0.7050

--- Try it yourself! ---
❓ My Question: Who developed BERT?
✅ Answer: 'Google' with score 0.9778


⚠️ A Quick Word on Model Limitations
It's important to understand that the BERT model we're using is not a general-purpose chatbot. It has been fine-tuned for a very specific task: extractive question answering.

This means its only job is to find and extract the exact span of text from the context that best answers the question. It does not understand abstract concepts, perform reasoning, or generate new sentences.

Watch what happens when we ask it about a joke.

In [8]:
# Example of another question
my_question = "Can BERT tell jokes?"
my_result = qa_pipeline(question=my_question, context=context)

print(f"❓ My Question: {my_question}")
print(f"✅ Answer: '{my_result['answer']}' with score {my_result['score']:.4f}")

❓ My Question: Can BERT tell jokes?
✅ Answer: 'Bidirectional Encoder Representations from Transformers' with score 0.0830


In [9]:
# The context contains a simple joke
joke_context = "I told my computer I needed a break, and now it won’t stop sending me Kit-Kat ads."

# The question requires understanding humor, not just extracting text
joke_question = "Why is this joke funny?"

# Let's see how the extractive model handles it
joke_result = qa_pipeline(question=joke_question, context=joke_context)

print(f"❓ Question: {joke_question}")
print(f"✅ Answer: '{joke_result['answer']}'")
print(f"Confidence Score: {joke_result['score']:.4f}")

❓ Question: Why is this joke funny?
✅ Answer: 'I needed a break'
Confidence Score: 0.0999


As you can see, the model fails. It extracts a piece of the sentence but completely misses the pun and the humorous concept. The low confidence score also shows it is struggling.

This model will generally fail at questions that require:

Reasoning or Synthesis: It can't combine multiple pieces of information.

"Yes/No" Answers: Unless the words "yes" or "no" are explicitly in the text.

Abstract Concepts: Like humor, sarcasm, or summarizing the main idea.

This is a key difference between an extractive model like this one and a generative AI like BARD, which is designed to understand these nuances and generate new, explanatory text.

### BERT Question-Answering Observations

The extractive question-answering model correctly answered factual questions when the information was available in the context. It identified the full meaning of BERT with a confidence score of 0.7050 and Google as its developer with a score of 0.9778.

However, when asked whether BERT can tell jokes, the model returned an unrelated answer with a low confidence score of 0.0830. It also failed to explain the computer joke, extracting "I needed a break" with a confidence score of 0.0999.

These results show that extractive question-answering models work well when an answer can be found directly in the context but may produce irrelevant answers when a question requires reasoning or information not provided in the text. Confidence scores should not be treated as guarantees of correctness.

### Part 7: Challenges

You have now seen a complete introductory NLP pipeline in action. But real NLP systems are rarely “finished.”  
There is almost always a missing piece, a trade-off, or a design decision that could be improved.

In this section, you will tackle **four beginner-friendly challenges** that push the current notebook a little further.  
Each challenge asks you to investigate a limitation, improve part of the pipeline, or test whether a different idea works better.

These are not just coding exercises — they are the kind of questions people ask when building real machine learning systems:

- Are we removing useful information by accident?
- Are we simplifying words too aggressively?
- How can we turn processed text into features a model could actually use?
- How does tokenization affect cost when we use commercial LLMs?

Take your time, experiment, and discuss what you notice.

#### Challenge 1: Are We Throwing Away Important Words?

Our current pipeline removes common stop words such as **"the"**, **"is"**, and **"and"**.  
That usually helps reduce noise, but it can also create a problem.

Some so-called “small” words actually carry **big meaning**.  
For example, the sentence:

> “This movie is **not** good.”

changes completely if we remove the word **not**.

Your challenge is to improve the stop-word removal step so that it keeps a few important **negation words** such as:

- `not`
- `no`
- `never`

Then test the result on a short sentence and see how the meaning changes when these words are preserved.

**Goal:** make the pipeline a little smarter by keeping words that are small, but important.

In [10]:
# Challenge 1 Scaffold:
# Modify stop-word removal so that important negation words are preserved.

test_sentence = "This movie is not good, and I would never recommend it to anyone."

def normalize_and_remove_stops_keep_negation(tokens):
    """
    Normalize tokens and remove stopwords while
    preserving important negation words.
    """
    normalized_tokens = [token.lower() for token in tokens]

    negation_words = {"not", "no", "nor", "never"}

    filtered_stopwords = set(stopwords.words("english")) - negation_words

    cleaned_tokens = [
        token for token in normalized_tokens
        if token not in filtered_stopwords
    ]

    return cleaned_tokens

# Try the original pipeline logic on the test sentence
original_tokens = simple_tokenizer(test_sentence)
original_cleaned = normalize_and_remove_stops(original_tokens)

# Try your improved version
improved_cleaned = normalize_and_remove_stops_keep_negation(original_tokens)

print("--- Original cleaned tokens ---")
print(original_cleaned)

print("\n--- Improved cleaned tokens (keep negation) ---")
print(improved_cleaned)

# Challenge 1 completed.
# The improved function preserves "not", which the original removed.
# Detailed observations are documented in the Markdown cell below.

--- Original cleaned tokens ---
['movie', 'good', 'would', 'never', 'recommend', 'anyone']

--- Improved cleaned tokens (keep negation) ---
['movie', 'not', 'good', 'would', 'never', 'recommend', 'anyone']


### Talking Points  — Challenge 1



**1. What words disappeared in the original version?**

The original preprocessing removed common stopwords such as "this", "is", "not", "and", "I", "it", and "to". Removing "not" changed the meaning of the original sentence.

**2. Which negation words did the improved version preserve?**

The improved version preserved "not" and "never". In our test, "never" was already present in the original output, while "not" was restored by the improved function.

**3. How could removing words like "not" or "never" confuse a sentiment analysis model?**

Removing negation words can change the meaning of a sentence. For example, "not good" may become "good", which could cause a sentiment analysis model to classify a negative review as positive.

**4. Which lines of code were most important?**

The most important lines were:

`negation_words = {"not", "no", "nor", "never"}`

`filtered_stopwords = set(stopwords.words("english")) - negation_words`

These lines ensure that important negation words are excluded from the stopword removal list.

#### Challenge 2: Is Stemming Too Aggressive?

Our notebook currently uses **stemming** to reduce words to a simpler form.  
That is useful, but stemming can sometimes be a little rough.

For example:

- `studies` might become `studi`
- `running` might become `run`
- `better` may not behave the way you expect

This raises a good beginner question:

> Is stemming always the best choice?

A more language-aware alternative is **lemmatization**, which tries to convert words into a proper dictionary base form.

Your challenge is to compare **stemming** and **lemmatization** on a small list of words and reflect on which output seems easier for humans to interpret.

**Goal:** explore the trade-off between a fast, simple method and a more linguistically meaningful method.

In [11]:
# Challenge 2 Scaffold:
# Compare stemming and lemmatization on a small set of example words.

# If needed, uncomment the following line once:
# import nltk
# nltk.download('wordnet')
# nltk.download('omw-1.4')

from nltk.stem import WordNetLemmatizer

example_words = ["running", "studies", "better", "cars", "wolves", "playing"]

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_words = []
lemmatized_words = []

# Apply stemming to each example word
for word in example_words:
    stemmed_words.append(stemmer.stem(word))

# Apply lemmatization to each example word
for word in example_words:
    lemmatized_words.append(lemmatizer.lemmatize(word))

print("--- Original Words ---")
print(example_words)

print("\n--- Stemmed Words ---")
print(stemmed_words)

print("\n--- Lemmatized Words ---")
print(lemmatized_words)

# Challenge 2 completed.
# Comparison and observations are documented in the Markdown cell below.

--- Original Words ---
['running', 'studies', 'better', 'cars', 'wolves', 'playing']

--- Stemmed Words ---
['run', 'studi', 'better', 'car', 'wolv', 'play']

--- Lemmatized Words ---
['running', 'study', 'better', 'car', 'wolf', 'playing']


### Talking Points  — Challenge 2



**1. Which words looked strange after stemming?**

The words "studies" and "wolves" became "studi" and "wolv". These are not standard English words because stemming removes endings using simple rules.

**2. Which words looked more natural after lemmatization?**

Lemmatization produced "study" from "studies" and "wolf" from "wolves". These results are proper English words. However, "running" and "playing" remained unchanged because the lemmatizer uses nouns by default.

**3. Why might a data scientist still choose stemming even if the output looks less polished?**

Stemming is generally faster and simpler because it uses rules to remove word endings. It can reduce vocabulary size without needing detailed linguistic information, making it useful for some large text datasets.

**4. What did this challenge teach you about trade-offs in NLP?**

This challenge showed that stemming and lemmatization have different advantages. Stemming is simpler and often faster, but it can produce unnatural words. Lemmatization produces more meaningful dictionary words, but its results depend on grammatical information. The best method depends on the NLP task.

#### Challenge 3: How Do We Turn Cleaned Text into Features for Machine Learning?

So far, we have cleaned text and produced processed tokens.  
That is an important step, but machine learning models still need **numbers**, not just words.

One common beginner-friendly solution is a **Bag-of-Words** representation.  
The idea is simple:

- count how often words appear
- choose the most common terms
- represent each document using those counts

This is one of the classic bridges between **NLP** and **Machine Learning**.

Your challenge is to build a small Bag-of-Words feature view from the processed dataset and inspect the most frequent tokens.

**Goal:** connect text preprocessing to the kind of numerical features a model could actually learn from.

In [12]:
# Challenge 3 Scaffold:
# Build a simple Bag-of-Words style summary from processed text.

from collections import Counter

# Reuse a small subset of documents so the output stays manageable
small_corpus = newsgroups_data.data[:50]

processed_documents = []

for doc in small_corpus:
    # Process each document using the existing NLP pipeline
    processed_tokens = nlp_pipeline(doc)
    processed_documents.append(processed_tokens)

# Flatten the processed documents into one list of tokens
all_tokens = []

for token_list in processed_documents:
    all_tokens.extend(token_list)

# Count how often each processed token appears
token_counts = Counter(all_tokens)

# Get the 15 most frequent tokens
top_15 = token_counts.most_common(15)

print("--- Top 15 Tokens in the Small Corpus ---")
print(top_15)

# Optional extension: Create a Bag-of-Words vector
# using the five most frequent tokens.

# Select the first processed document
selected_doc = processed_documents[0]

# Select the top five most frequent tokens
vocabulary = [token for token, count in top_15[:5]]

# Count the tokens in the selected document
document_counts = Counter(selected_doc)

# Create the Bag-of-Words vector
bow_vector = {
    word: document_counts[word]
    for word in vocabulary
}

print("\n--- Example Bag-of-Words Vector ---")
print(bow_vector)

# Find a document containing at least one vocabulary word
example_index = next(
    (i for i, tokens in enumerate(processed_documents)
     if any(word in tokens for word in vocabulary)),
    None
)

if example_index is not None:
    example_counts = Counter(processed_documents[example_index])

    example_vector = {
        word: example_counts[word]
        for word in vocabulary
    }

    print("\n--- Non-Zero Bag-of-Words Example ---")
    print("Document index:", example_index)
    print(example_vector)

--- Top 15 Tokens in the Small Corpus ---
[('det', 45), ('nyr', 40), ('tor', 39), ('bo', 37), ('mtl', 33), ('la', 30), ('chi', 30), ('order', 26), ('one', 25), ('pit', 24), ('time', 23), ('use', 22), ('edm', 22), ('phi', 21), ('1', 20)]

--- Example Bag-of-Words Vector ---
{'det': 0, 'nyr': 0, 'tor': 0, 'bo': 0, 'mtl': 0}

--- Non-Zero Bag-of-Words Example ---
Document index: 7
{'det': 0, 'nyr': 0, 'tor': 0, 'bo': 2, 'mtl': 0}


### Talking Points  — Challenge 3



**1. Which processed words appeared most often?**

The most frequent tokens were "det" (45), "nyr" (40), "tor" (39), "bo" (37), and "mtl" (33). These were the five most common tokens across the 50 processed documents.

**2. Did the most common words seem meaningful, surprising, or too generic?**

Some tokens, such as "det", "nyr", and "tor", appear to be abbreviations, possibly referring to sports teams. Their high frequency was surprising because they are not general English words. Other frequent words, such as "order", "time", and "use", are more recognizable but can be generic. This shows that frequent tokens are not always the most informative features.

**3. How does a Bag-of-Words representation help a machine learning model?**

Bag-of-Words converts text into numerical word counts. This allows machine learning models to work with text data. For example, our first document had zero counts for all five selected words, while document 7 contained the token "bo" twice.

**4. What information is lost when we reduce a document to word counts?**

Bag-of-Words does not preserve word order, sentence structure, or the full context of a document. Two sentences with different meanings can have similar word counts. Therefore, Bag-of-Words is useful for representing word frequency but has limitations when understanding language meaning.

#### Challenge 4: Why Does Tokenization Matter for LLM Billing?

When we use a commercial Large Language Model, we are usually **not billed per prompt** or per paragraph.  
We are billed by **tokens**.

A token is a chunk of text. Sometimes it is a whole word, sometimes part of a word, and sometimes punctuation.  
That means two prompts that look very similar to a human can still produce **different token counts, costs, and response times**.

This makes tokenization more than just a preprocessing step.  
It becomes part of a real engineering decision:

- How much text should we send?
- Should we remove stop words?
- Should we simplify words first?
- Does a Bag-of-Words view reduce meaning too much?
- How much extra cost comes from the **prompt** compared with the **model's answer**?

For this reflection challenge, imagine you are preparing a long educational prompt for a commercial LLM.  
Use the prompt below as your sample text, then explore how the token count changes if you:

1. remove stop words  
2. apply stemming or lemmatization  
3. convert the text into a Bag-of-Words style view  

You may also compare your observations with external tools such as:

- `https://pricepertoken.com/`
- `https://pricepertoken.com/token-counter`

A real example shared for this workshop showed that two models could receive almost the same prompt, yet differ a lot in **completion length, latency, and cost**.  
That is a useful reminder: **tokenization affects billing, but the model's response style matters too.**

**Goal:** reflect on how preprocessing choices may reduce token counts, and also think critically about what is lost when we compress a prompt too aggressively.


In [13]:

# Challenge 4: LLM Tokenization and Cost
# Explore how NLP preprocessing affects token counts and estimated costs.

from collections import Counter
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Original workshop prompt
workshop_prompt = """
I am a professor of Machine Learning at the college level. You are an experienced
Machine Learning Engineer and Data Scientist. I am developing a workshop on time series
using Jupyter Notebooks as a project to help train my students in an active learning environment.

In this case, I want them to learn Data and Time Series Analysis, the Moving Averages calculation,
Decomposition Models, how to Deseasonalize a Time Series, Autoregressive Models, ARMA Models,
ARIMA Models, the Augmented Dickey-Fuller test, and Machine Learning application design patterns,
all simultaneously.

The workshop is based on active learning and requires them to read, brainstorm, write code,
reflect on model performance, and compare results with their peers.
""".strip()


# 1. Simple tokenizer
# This is a classroom word tokenizer, not a commercial LLM tokenizer.
def simple_tokenize(text):
    """Split text into lowercase word tokens."""
    tokens = re.findall(r"\b\w+\b", text.lower())
    return tokens


# 2. Stop-word removal
def remove_stop_words(tokens):
    """Remove common stopwords while preserving negation words."""
    stop_words = set(stopwords.words("english"))

    # Preserve important negation words from Challenge 1
    negation_words = {"not", "no", "nor", "never"}
    filtered_stopwords = stop_words - negation_words

    filtered_tokens = [
        token for token in tokens
        if token not in filtered_stopwords
    ]

    return filtered_tokens


# 3. Stemming
def apply_stemming(tokens):
    """Apply Porter stemming to each token."""
    stemmer = PorterStemmer()

    stemmed_tokens = [
        stemmer.stem(token) for token in tokens
    ]

    return stemmed_tokens


# 4. Bag-of-Words representation
def bag_of_words_view(tokens):
    """Count the frequency of each processed token."""
    bow = Counter(tokens)
    return bow


# --- Baseline ---
baseline_tokens = simple_tokenize(workshop_prompt)

print("--- Baseline ---")
print("Baseline token count:", len(baseline_tokens))
print("First 25 baseline tokens:", baseline_tokens[:25])


# --- Version A: Stop-word removal ---
tokens_no_stops = remove_stop_words(baseline_tokens)

print("\n--- Stop-word Removal ---")
print("Token count after stop-word removal:", len(tokens_no_stops))


# --- Version B: Stemming ---
stemmed_tokens = apply_stemming(tokens_no_stops)

print("\n--- Stemming ---")
print("Token count after stemming:", len(stemmed_tokens))


# --- Version C: Bag-of-Words ---
bow_counts = bag_of_words_view(stemmed_tokens)

top_15 = bow_counts.most_common(15)

print("\n--- Top 15 Bag-of-Words Terms ---")
print(top_15)


# --- Optional Extension: Hypothetical Cost Estimation ---
# Illustrative rate, not an actual provider price.
price_per_1k_tokens = 0.008

baseline_cost = (
    len(baseline_tokens) / 1000 * price_per_1k_tokens
)

processed_cost = (
    len(stemmed_tokens) / 1000 * price_per_1k_tokens
)

cost_savings = baseline_cost - processed_cost

reduction_percentage = (
    (len(baseline_tokens) - len(stemmed_tokens))
    / len(baseline_tokens) * 100
    if baseline_tokens else 0
)

print("\n--- Hypothetical Cost Comparison ---")
print(f"Estimated baseline prompt cost: ${baseline_cost:.6f}")
print(f"Estimated processed prompt cost: ${processed_cost:.6f}")
print(f"Estimated cost savings: ${cost_savings:.6f}")
print(f"Token count reduction: {reduction_percentage:.2f}%")


# --- Reflection ---
print("\n--- Reflection ---")

print(
    f"Stop-word removal reduced the token count from "
    f"{len(baseline_tokens)} to {len(tokens_no_stops)}, "
    f"while stemming changed word forms without reducing "
    f"the number of tokens."
)

print(
    "Although removing words reduces the word-token count, "
    "it may also remove important context and grammatical meaning. "
    "Real LLM costs depend on the model's own tokenizer, so "
    "these calculations are only classroom estimates."
)


--- Baseline ---
Baseline token count: 114
First 25 baseline tokens: ['i', 'am', 'a', 'professor', 'of', 'machine', 'learning', 'at', 'the', 'college', 'level', 'you', 'are', 'an', 'experienced', 'machine', 'learning', 'engineer', 'and', 'data', 'scientist', 'i', 'am', 'developing', 'a']

--- Stop-word Removal ---
Token count after stop-word removal: 71

--- Stemming ---
Token count after stemming: 71

--- Top 15 Bag-of-Words Terms ---
[('learn', 6), ('model', 5), ('machin', 3), ('time', 3), ('seri', 3), ('data', 2), ('workshop', 2), ('activ', 2), ('professor', 1), ('colleg', 1), ('level', 1), ('experienc', 1), ('engin', 1), ('scientist', 1), ('develop', 1)]

--- Hypothetical Cost Comparison ---
Estimated baseline prompt cost: $0.000912
Estimated processed prompt cost: $0.000568
Estimated cost savings: $0.000344
Token count reduction: 37.72%

--- Reflection ---
Stop-word removal reduced the token count from 114 to 71, while stemming changed word forms without reducing the number of tok

### Talking Points  — Challenge 4



**1. How many tokens did your baseline prompt contain?**

The baseline prompt contained 114 tokens using our simple word tokenizer.

**2. If stop words were removed, how much shorter did the prompt become?**

After removing stopwords, the token count decreased from 114 to 71. This removed 43 tokens, representing a 37.72% reduction.

**3. Did stemming or lemmatization reduce the token count much more, or mostly change the shape of the words?**

Stemming did not reduce the number of tokens. The count remained 71. It mainly changed word forms, such as "learning" to "learn" and "machine" to "machin". Lemmatization was not tested in this challenge.

**4. What happened when you converted the text into a Bag-of-Words view?**

Bag-of-Words converted the processed text into word-frequency counts. The most frequent terms included "learn" (6), "model" (5), and "machin" (3). This representation is useful for numerical analysis but loses word order and sentence structure.

**5. Would you actually send the Bag-of-Words version to a commercial LLM? Why or why not?**

No. A Bag-of-Words representation loses grammar, word order, and much of the original meaning. A commercial LLM generally needs meaningful context to understand instructions and generate useful responses.

**6. In your opinion, what matters more for billing in the example above: the prompt tokens or the completion tokens?**

Our experiment measured only the prompt tokens, so they directly affected our estimated cost. In real commercial LLM applications, both input (prompt) and output (completion) tokens can affect billing. Which costs more depends on their quantities and the model's pricing.

**7. Which preprocessing step saved the most tokens without damaging the meaning too much?**

Stopword removal saved the most tokens, reducing the count by 43. Many important technical terms remained, including "model", "data", and "time". However, removing stopwords can sometimes change meaning, so important words such as negations should be preserved. Stemming did not provide additional token-count savings in this experiment.

**Overall Observation**

Our NLP preprocessing reduced the workshop's word-token count by 37.72%, giving a hypothetical cost reduction from $0.000912 to $0.000568. However, fewer word tokens do not necessarily mean lower real LLM costs or better results. Actual token counts depend on the model's tokenizer, and excessive preprocessing can damage the meaning of the prompt.
